In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import hashlib
import osmnx as ox
from shapely import Point

In [ ]:
# Load the dataset

#MethaneSAT Scenes
msat_scenes = gpd.read_file("data/msat_all_scenes.gpkg")

##Manually validated identified tanks
ordos12_output = gpd.read_file("data/Ordos3_manually_validated.gpkg")
ordos3_output = gpd.read_file("data/Ordos12_manually_validated.gpkg")


In [ ]:
# Functions for calculations

def get_radius_est(poly):

    # Extract the exterior corner coordinates
    x, y = poly.exterior.coords.xy

    # Calculate the lengths of two adjacent sides
    side1 = Point(x[0], y[0]).distance(Point(x[1], y[1]))
    side2 = Point(x[1], y[1]).distance(Point(x[2], y[2]))

    # Return the smaller of the two sides
    short_side = min(side1, side2) /2

    return short_side

def get_area_est(poly):

    # Extract the exterior corner coordinates
    x, y = poly.exterior.coords.xy

    # Calculate the lengths of two adjacent sides
    side1 = Point(x[0], y[0]).distance(Point(x[1], y[1]))
    side2 = Point(x[1], y[1]).distance(Point(x[2], y[2]))

    # Return the smaller of the two sides, divide by 2 to get ragius
    short_side_radius = min(side1, side2) / 2

    area_estimate = 3.1415926 * (short_side_radius**2)

    return area_estimate

In [ ]:

def process_geodataframe_tank(gdf, crs_epsg, source_name, feature_type, location):
    """
    Projects a GeoDataFrame to a specific UTM CRS, calculates area in km2,
    adds lat/lon centroids, and sets Source/Type attributes.
    """
    # Create a copy to avoid changing original data
    gdf_processed = gdf.copy()
    
    #Project to target CRS (with meters as unit to avoid getting area in degrees)
    gdf_processed = gdf_processed.to_crs(crs_epsg)
    
    # Calculate area in km2
    gdf_processed['radius_est'] = gdf_processed['geometry'].apply(get_radius_est)
    gdf_processed['area_est'] = gdf_processed['geometry'].apply(get_area_est)
    
    # Calculate centroids (lat/lon) in WGS84 
    centroids = gdf_processed.centroid.to_crs(epsg=4326)
    gdf_processed['lat'] = centroids.y
    gdf_processed['lon'] = centroids.x
    
    # Project back to WGS84 for consistent output
    gdf_processed = gdf_processed.to_crs(epsg=4326)
    
    # Add data sources and types
    gdf_processed['Source'] = source_name #How data was sourced
    gdf_processed['Type'] = feature_type #type of feature
    gdf_processed['location'] = location #msat scene


In [ ]:
##process gdfs and combine into one

ordos12_output_processed = process_geodataframe_tank(ordos12_output, 'EPSG:32649', "ML", "Tank", 'Ordos')

ordos3_output_processed = process_geodataframe_tank(ordos3_output, 'EPSG:32649', "ML", "Tank", 'Ordos')

ordos_123 = gpd.GeoDataFrame(pd.concat([ordos12_output_processed, ordos3_output_processed], ignore_index=True), crs=ordos12_output_processed.crs)

In [ ]:
#Adding spatial join nearest for polygons outside the bounds of MethaneSAT scenes

# 1. Initial spatial join
ordos_123_loc = gpd.sjoin(ordos_123, msat_scenes, how="left", predicate="within")

# 2. Separate matched and unmatched records
unmatched = ordos_123_loc[ordos_123_loc['index_right'].isna()].copy()
matched = ordos_123_loc[~ordos_123_loc['index_right'].isna()].copy()

# 3. Find the overlapping columns that caused the suffixes
# We exclude the geometry and the join index to see what's duplicating
left_cols = set(ordos_123.columns) - {'geometry'}
right_cols = set(msat_scenes.columns) - {'geometry'}
overlapping_cols = left_cols.intersection(right_cols)

# 4. Clean up unmatched to prevent double-suffixing (_left_left, etc.)
# Drop columns that came from msat_scenes in the first join
cols_to_drop = list(right_cols) + ['index_right']
unmatched_cleaned = unmatched.drop(columns=cols_to_drop, errors='ignore')

# 5. Nearest match for unmatched (with suffix cleanup)
nearest_match = gpd.sjoin_nearest(unmatched_cleaned, msat_scenes, how='left')

# 6. Final Result
final_result = gpd.pd.concat([matched, nearest_match], ignore_index=True)

# 7. If any duplicate columns still slip through, drop them
final_result = final_result.loc[:, ~final_result.columns.duplicated()]

/opt/anaconda3/envs/spatial/lib/python3.12/site-packages/geopandas/array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


In [ ]:
ordos_123_tank_est = gpd.GeoDataFrame(
    final_result, geometry=gpd.points_from_xy(final_result.lon, final_result.lat), crs="EPSG:4326"
)
ordos_123_tank_est = ordos_123_tank_est.to_crs('EPSG:32649')

ordos_123_tank_est['geometry'] = ordos_123_tank_est.buffer(ordos_123_tank_est['radius_est'])

manual_check_map = {
    'no issue': 'Correct',
    'fixed': 'Flag'
}

ordos_123_tank_est['manual_check'] = ordos_123_tank_est['manual_check'].map(manual_check_map)

sjoin_check_map = {
    'SHANXI: 1': 'ORDOS: 3',
}

ordos_123_tank_est['name'] = ordos_123_tank_est['name'].map(sjoin_check_map)


ordos_123_tank_est_subset = ordos_123_tank_est[['manual_check', 'geometry', 'radius_est', 'area_est', 'lat', 'lon', 'name']]


In [ ]:
#writing to gpkg and xlsx

ordos_123_tank_est_subset.to_file("ordos_123.gpkg", driver="GPKG", mode='w',encoding='utf-8')
ordos_123_tank_est_subset.to_excel('ordos_123_tanks.xlsx')